In [ ]:
from __future__ import annotations
from typing import Any
import egglog


class Wire(egglog.Expr):
    @classmethod
    def from_input(cls, name: egglog.StringLike) -> Wire: ...

    @classmethod
    def from_dff(cls, dff_tag: egglog.StringLike) -> Wire: ...
    # TODO: use relation (from q d)
    # for example:
    # @classmethod
    # def from_d(cls, d: Wire) -> Wire: ...

    def __and__(self, other: Wire) -> Wire: ...

    def __or__(self, other: Wire) -> Wire: ...

    def __xor__(self, other: Wire) -> Wire: ...

    def __invert__(self) -> Wire: ...

class WireVec(egglog.Expr):
    @classmethod
    def from_inputs(cls, name: egglog.StringLike, width: egglog.i64Like) -> WireVec: ...

    @classmethod
    def from_wires(cls, wires: egglog.Vec[Wire]) -> WireVec: ...

    @classmethod
    def add(cls, out_width: egglog.i64Like, a: WireVec, b: WireVec) -> WireVec: ...

    @classmethod
    def mul(cls, out_width: egglog.i64Like, a: WireVec, b: WireVec) -> WireVec: ...

    def __getitem__(self, index: egglog.i64Like) -> Wire: ...   # this is necessary for indexing from_inputs

class FromDffsRelation(egglog.Expr):
    # vec[cur:] = dff_tags
    def __init__(self, cur: egglog.i64Like, vec: egglog.Vec[Wire], dff_tags: egglog.Vec[egglog.String]): ...

class CanRetimed(egglog.Expr):
    def __init__(self, wv: WireVec): ...

egraph = egglog.EGraph()

dff0 = egraph.let("dff0", Wire.from_dff("dff0"))
dff1 = egraph.let("dff1", Wire.from_dff("dff1"))
dff2 = egraph.let("dff2", Wire.from_dff("dff2"))
dff3 = egraph.let("dff3", Wire.from_dff("dff3"))
dff4 = egraph.let("dff4", Wire.from_dff("dff4"))
dff5 = egraph.let("dff5", Wire.from_dff("dff5"))
dff6 = egraph.let("dff6", Wire.from_dff("dff6"))
dff7 = egraph.let("dff7", Wire.from_dff("dff7"))
a = egraph.let("a", WireVec.from_wires(egglog.Vec(dff0, dff1, dff2, dff3)))
b = egraph.let("b", WireVec.from_wires(egglog.Vec(dff4, dff5, dff6, dff7)))
c = egraph.let("c", WireVec.add(5, a, b))

i = egglog.var("i", egglog.i64)
vec0, vec1 = egglog.vars_("vec0 vec1", egglog.Vec[Wire])
strvec0, strvec1 = egglog.vars_("strvec0 strvec1", egglog.Vec[egglog.String])
str0 = egglog.var("str0", egglog.String)
wv0 = egglog.var("wv0", WireVec)

egraph.register(
    egglog.rule(    # base case
        WireVec.add(i, WireVec.from_wires(vec0), WireVec.from_wires(vec1))
    ).then(
        FromDffsRelation(i - 1, vec0, egglog.Vec[egglog.String]()),
        FromDffsRelation(i - 1, vec1, egglog.Vec[egglog.String]())
    ),
    egglog.rule(    # inductive case
        FromDffsRelation(i, vec0, strvec0),
        i > 0,
        egglog.eq(Wire.from_dff(str0)).to(vec0[i - 1])
    ).then(
        egglog.subsume(FromDffsRelation(i, vec0, strvec0)),
        FromDffsRelation(i - 1, vec0, strvec0.push(str0))
    ),
    egglog.rule(
        egglog.eq(wv0).to(WireVec.add(i, WireVec.from_wires(vec0), WireVec.from_wires(vec1))),
        FromDffsRelation(0, vec0, strvec0),
        FromDffsRelation(0, vec1, strvec1)
    ).then(
        CanRetimed(wv0)
    )
)

egraph.run(100)

egraph.display(graphviz=True)

In [1]:
import emap
import json
import time

with open("bad_multiplier.json", "r") as f:
    mod = json.load(f)["modules"]["top"]

netlist = emap.Netlist()

start_time = time.time()
netlist.build_from_json(mod, clk="clk")
print(f"Netlist built in {time.time() - start_time:.2f} seconds")

start_time = time.time()
netlist.run((emap.logic_rules + emap.arith_rules + emap.retiming_rules) * 100)
print(f"Netlist saturated in {time.time() - start_time:.2f} seconds")

netlist.display(graphviz=True)
# for name, wv in netlist.outputs.items():
#     print(f"{name}: {wv}")

AssertionError: Can only subsume calls, not literals or vars